# DOTM Xbox 360 ISO — облачный список файлов
Запустите единственную ячейку. Список файлов будет записан в `DOTM_ISO_MANIFEST.txt` на Google Drive. ISO не копируется и не распаковывается.


In [ ]:
from google.colab import drive
import os, subprocess, pathlib
drive.mount('/content/drive')
iso = pathlib.Path('/content/drive/MyDrive/Transformers - Dark of the Moon (USA) (En,Fr,De,Es,It).iso')
if not iso.is_file():
    raise FileNotFoundError(f'ISO not found at {iso}. Check the file is in My Drive root.')
print('ISO found:', iso.name, iso.stat().st_size, 'bytes')
source = pathlib.Path('/content/extract-xiso')
if not source.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/XboxDev/extract-xiso.git',str(source)],check=True)
subprocess.run(['cmake','-S',str(source),'-B',str(source/'build'),'-DCMAKE_BUILD_TYPE=Release'],check=True)
subprocess.run(['cmake','--build',str(source/'build'),'-j','2'],check=True)
exe=next((p for p in (source/'build').rglob('extract-xiso') if p.is_file() and os.access(p,os.X_OK)),None)
if exe is None:
    raise RuntimeError('extract-xiso executable was not built')
result=subprocess.run([str(exe),'-l',str(iso)],capture_output=True,text=True)
manifest=pathlib.Path('/content/drive/MyDrive/DOTM_ISO_MANIFEST.txt')
manifest.write_text('ISO: '+iso.name+'\nSIZE: '+str(iso.stat().st_size)+' bytes\nLIST EXIT CODE: '+str(result.returncode)+'\n\n'+result.stdout+'\nSTDERR:\n'+result.stderr,encoding='utf-8')
print('Saved manifest:',manifest)
print('\n'.join((result.stdout or result.stderr).splitlines()[:80]))
if result.returncode:
    print('Indexing reported an error; the complete output is saved for diagnosis.')
